# TD-PINNs: Burgers Equation (1D & 2D)

Time-Decomposition Physics-Informed Neural Networks for the viscous Burgers equation.
A single `TD_PINNS` class handles both 1-D and 2-D cases.  
The PDE residual is injected via the `pde_loss_fn` parameter — plug in any custom equation.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import matplotlib.pyplot as plt
import os
import pandas as pd

## Core Model

Built-in loss functions for 1-D and 2-D Burgers, plus the unified `TD_PINNS` network.

In [ ]:
# ── Built-in PDE loss functions ────────────────────────────────────────────

def burgers_1d_loss(model, X, u0, u1):
    # 1-D viscous Burgers: du/dt + u*du/dx = nu*d2u/dx2
    du_dx = torch.autograd.grad(
        u1, X, grad_outputs=torch.ones_like(u1),
        retain_graph=True, create_graph=True)[0]
    du_dxx = torch.autograd.grad(
        du_dx, X, grad_outputs=torch.ones_like(du_dx),
        retain_graph=True, create_graph=True)[0]
    res = (u1 - u0) / model.dt + u1 * du_dx - model.nu * du_dxx
    return torch.mean(res**2), res


def burgers_2d_loss(model, X, u0, u1):
    # 2-D viscous Burgers: du/dt + u*(du/dx + du/dy) = nu*(d2u/dx2 + d2u/dy2)
    g = torch.autograd.grad(
        u1, X, grad_outputs=torch.ones_like(u1),
        retain_graph=True, create_graph=True)[0]
    du_dx, du_dy = g[:, 0:1], g[:, 1:2]
    du_dxx = torch.autograd.grad(
        du_dx, X, grad_outputs=torch.ones_like(du_dx),
        retain_graph=True, create_graph=True)[0][:, 0:1]
    du_dyy = torch.autograd.grad(
        du_dy, X, grad_outputs=torch.ones_like(du_dy),
        retain_graph=True, create_graph=True)[0][:, 1:2]
    res = (u1 - u0) / model.dt + u1 * (du_dx + du_dy) - model.nu * (du_dxx + du_dyy)
    return torch.mean(res**2), res


# ── Network ────────────────────────────────────────────────────────────────

class TD_PINNS(nn.Module):
    # Time-Decomposition PINN.
    # x_dim       : spatial dimension (1 or 2)
    # pde_loss_fn : callable(model, X, u0, u1) -> (scalar_loss, residual)
    #               Defaults to the built-in Burgers loss for x_dim.

    def __init__(self, device, x_dim=1, hid=64, u_dim=1,
                 nu=0.3, dt=0.01, pde_loss_fn=None):
        super().__init__()
        self.device = device
        self.x_dim  = x_dim
        self.nu, self.dt = nu, dt
        self.net = nn.Sequential(
            nn.Linear(x_dim, hid), nn.Tanh(),
            nn.Linear(hid, hid),   nn.Tanh(),
            nn.Linear(hid, hid),   nn.Tanh(),
            nn.Linear(hid, u_dim),
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=1.)
                nn.init.zeros_(m.bias)
        self.pde_loss_fn = pde_loss_fn or (
            burgers_1d_loss if x_dim == 1 else burgers_2d_loss
        )
        self.to(device)

    def forward(self, x):
        return self.net(x)

    def loss_initial(self, X, u0_real, x_bc, bc):
        u0       = self.forward(X)
        loss_IC  = nn.functional.mse_loss(u0, u0_real)
        residual = u0**2 - u0_real**2
        loss_bc  = nn.functional.mse_loss(self.forward(x_bc), bc)
        return 10 * loss_IC + 10 * loss_bc, loss_IC, loss_bc, residual

    def loss(self, X, u0, x_bc, bc):
        u1 = self.forward(X)
        loss_pde, residual = self.pde_loss_fn(self, X, u0, u1)
        loss_bc = nn.functional.mse_loss(self.forward(x_bc), bc)
        return 0.1 * loss_pde + 10 * loss_bc, loss_pde, loss_bc, residual

    def grad(self, X):
        # Compute gradients for adaptive collocation strategies.
        # 1-D: returns (du_dx, du_dxx)
        # 2-D: returns (grad_u [N,2], laplacian [N,1])
        X = X.detach().requires_grad_(True)
        u = self.forward(X)
        if self.x_dim == 1:
            du_dx = torch.autograd.grad(
                u, X, grad_outputs=torch.ones_like(u),
                retain_graph=True, create_graph=True)[0]
            du_dxx = torch.autograd.grad(
                du_dx, X, grad_outputs=torch.ones_like(du_dx),
                retain_graph=True, create_graph=True)[0]
            return du_dx, du_dxx
        else:
            g = torch.autograd.grad(
                u, X, grad_outputs=torch.ones_like(u),
                retain_graph=True, create_graph=True)[0]
            du_dx, du_dy = g[:, 0:1], g[:, 1:2]
            du_dxx = torch.autograd.grad(
                du_dx, X, grad_outputs=torch.ones_like(du_dx),
                retain_graph=True, create_graph=True)[0][:, 0:1]
            du_dyy = torch.autograd.grad(
                du_dy, X, grad_outputs=torch.ones_like(du_dy),
                retain_graph=True, create_graph=True)[0][:, 1:2]
            return g, du_dxx + du_dyy

## Logger

In [ ]:
class Logger:
    def __init__(self, filename='data_res.csv'):
        self.filename = filename
        self.df = pd.read_csv(filename) if os.path.exists(filename) else pd.DataFrame()

    def add(self, method, time_vals, real, estimated, dt, nu, elapsed):
        new = pd.DataFrame({
            'method': method, 'time': time_vals,
            'real': real, 'estimated': estimated[:, 0],
            'dt': dt, 'nu': nu, 'time_iteration': elapsed,
        })
        self.df = pd.concat([self.df, new], ignore_index=True) if not self.df.empty else new
        os.makedirs(os.path.dirname(self.filename) or '.', exist_ok=True)
        self.df.to_csv(self.filename, index=False)

## Trainers

In [ ]:
def _optimize_device():
    torch.backends.cudnn.benchmark = torch.cuda.is_available()
    torch.set_float32_matmul_precision('medium')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()
    if torch.backends.mps.is_available():
        print('Using MPS'); return torch.device('mps')
    if torch.cuda.is_available():
        print('Using CUDA'); return torch.device('cuda')
    print('Using CPU'); return torch.device('cpu')


class TrainerBurger1D:

    def __init__(self):
        self.device = _optimize_device()

    def set_simulation_parameters(self, nu=0.07, t0=0.0, dt=0.1, nt=10, x_bc=None):
        self.nu, self.t0, self.dt, self.nt = nu, t0, dt, nt
        if x_bc is None:
            x_bc = torch.Tensor([0, 2 * np.pi])
        self.x_bc   = x_bc
        self.x_test = torch.linspace(x_bc[0], x_bc[1], 1000)

    def set_training_parameters(self, tol=1e-2, tol_ic=None, max_epochs=80000,
                                 n_prints=10, lr=1e-2, min_lr=1e-8, pde_loss_fn=None):
        self.tol, self.tol_ic = tol, (tol_ic or tol)
        self.max_epochs       = max_epochs
        self.n_prints         = n_prints
        self.lr, self.min_lr  = lr, min_lr
        self.pde_loss_fn      = pde_loss_fn

    def set_point_collocation_distribution(self, func):
        self.func_distribution = func

    def set_initial_condition(self, x_initial, func=None, u_initial=None):
        self.x_initial = x_initial
        if func:
            x_np           = x_initial.numpy() if torch.is_tensor(x_initial) else x_initial
            self.u_initial = torch.as_tensor(func(t=self.t0, x=x_np), dtype=torch.float32)
            bc_np          = self.x_bc.numpy()
            self.bc_initial = torch.as_tensor(func(t=self.t0, x=bc_np), dtype=torch.float32)
        else:
            self.u_initial  = torch.as_tensor(u_initial, dtype=torch.float32)
            self.bc_initial = self.u_initial[[0, -1]]

    def set_boundary_function(self, func):
        self.boundary_func = func

    def set_analytic_solution(self, func):
        self.analytic_func = func

    def set_device(self, name):
        self.device = torch.device(name)

    def train_td_implicit(self):
        self.PINNS_list  = []
        self.PINNS_times = []
        PINN_old, pde_res = self._train_iter(
            self.x_initial, self.u_initial,
            initial_condition=True, tol=self.tol_ic,
            x_bc=self.x_bc, bc=self.bc_initial
        )
        self.PINNS_list.append(PINN_old)
        for i in range(1, self.nt):
            t = self.t0 + i * self.dt
            print(f'\n=== Step {i}/{self.nt-1}  t={t:.4f} ===')
            start   = time.perf_counter()
            x_i     = self.func_distribution(PINN_old, t, pde_res).reshape(-1, 1)
            u_0     = PINN_old.forward(x_i)
            bc      = torch.as_tensor(
                self.boundary_func(t=t, x=self.x_bc.numpy()), dtype=torch.float32)
            PINN_i, pde_res = self._train_iter(
                x_i, u_0, tol=self.tol, previous_PINN=PINN_old,
                x_bc=self.x_bc, bc=bc
            )
            elapsed = time.perf_counter() - start
            self.PINNS_list.append(PINN_i)
            self.PINNS_times.append(elapsed)
            print(f'  Done in {elapsed:.2f}s')
            PINN_old = PINN_i

    def _train_iter(self, x_train, u_train, previous_PINN=None,
                    initial_condition=False, tol=1e-3, x_bc=None, bc=None):
        def to_t(a):
            a = a.detach() if torch.is_tensor(a) else a
            return torch.as_tensor(a, dtype=torch.float32).reshape(-1, 1).to(self.device)
        x_train = to_t(x_train).requires_grad_(True)
        u_train = to_t(u_train)
        x_bc    = to_t(x_bc).requires_grad_(True)
        bc      = to_t(bc)
        PINN = TD_PINNS(device=self.device, x_dim=1, nu=self.nu, dt=self.dt,
                        pde_loss_fn=getattr(self, 'pde_loss_fn', None))
        if previous_PINN:
            PINN.load_state_dict(previous_PINN.state_dict())
        opt   = torch.optim.AdamW(PINN.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode='min', patience=300, factor=0.5, min_lr=self.min_lr)
        print_every = max(1, self.max_epochs // self.n_prints)
        for i in range(self.max_epochs):
            opt.zero_grad()
            if initial_condition:
                loss, l_ic, l_bc, res = PINN.loss_initial(x_train, u_train, x_bc, bc)
            else:
                loss, l_pde, l_bc, res = PINN.loss(x_train, u_train, x_bc, bc)
            loss.backward(retain_graph=True)
            v = loss.item()
            if v <= tol:
                print(f'  Converged epoch={i}  loss={v:.6f}')
                break
            opt.step()
            sched.step(v)
            if i % print_every == 0:
                print(f'  Epoch {i:6d}: loss={v:.6f}  lr={opt.param_groups[0]["lr"]:.2e}')
        return PINN, res

    def plot_solution(self, PINN, t, nx=1000):
        x = torch.linspace(self.x_bc[0], self.x_bc[1], nx).reshape(-1, 1).to(self.device)
        with torch.no_grad():
            u = PINN.forward(x).cpu().numpy()
        plt.figure(figsize=(8, 4))
        plt.plot(x.cpu().numpy(), u, label=f'PINN  t={t:.3f}')
        if hasattr(self, 'analytic_func'):
            u_ex = self.analytic_func(t=t, x=x.cpu().numpy().ravel())
            plt.plot(x.cpu().numpy(), u_ex, '--', label='exact')
        plt.legend(); plt.xlabel('x'); plt.ylabel('u')
        plt.title(f'1-D Burgers  t={t:.4f}'); plt.grid(True); plt.show()

    def plot_evolution(self, nx=1000):
        for i, P in enumerate(self.PINNS_list):
            self.plot_solution(P, self.t0 + i * self.dt, nx)

In [ ]:
class TrainerBurger2D:

    def __init__(self):
        self.device = _optimize_device()

    def set_simulation_parameters(self, nu=0.01, t0=0.0, dt=0.2, nt=6,
                                   x_range=(-1., 1.), y_range=(-1., 1.)):
        self.nu, self.t0, self.dt, self.nt = nu, t0, dt, nt
        self.x_range, self.y_range = x_range, y_range

    def set_training_parameters(self, tol=1e-2, tol_ic=None, max_epochs=30000,
                                 n_prints=10, lr=1e-2, min_lr=1e-8, pde_loss_fn=None):
        self.tol, self.tol_ic = tol, (tol_ic or tol)
        self.max_epochs       = max_epochs
        self.n_prints         = n_prints
        self.lr, self.min_lr  = lr, min_lr
        self.pde_loss_fn      = pde_loss_fn

    def set_point_collocation_distribution(self, func):
        self.func_distribution = func

    def set_device(self, name):
        self.device = torch.device(name)

    def create_2d_grid(self, nx, ny):
        x = torch.linspace(self.x_range[0], self.x_range[1], nx)
        y = torch.linspace(self.y_range[0], self.y_range[1], ny)
        xx, yy = torch.meshgrid(x, y, indexing='ij')
        return torch.stack([xx.flatten(), yy.flatten()], dim=1).to(self.device)

    def get_boundary_points(self, n=50):
        xmn, xmx = self.x_range
        ymn, ymx = self.y_range
        bot = torch.stack([torch.linspace(xmn, xmx, n), torch.full((n,), ymn)], 1)
        top = torch.stack([torch.linspace(xmn, xmx, n), torch.full((n,), ymx)], 1)
        lft = torch.stack([torch.full((n,), xmn), torch.linspace(ymn, ymx, n)], 1)
        rgt = torch.stack([torch.full((n,), xmx), torch.linspace(ymn, ymx, n)], 1)
        return torch.cat([bot, top, lft, rgt]).to(self.device)

    def set_initial_condition(self, xy_initial, func=None, u_initial=None):
        self.xy_initial = xy_initial.to(self.device)
        if func:
            self.initial_func = func
            x = xy_initial[:, 0].cpu().numpy()
            y = xy_initial[:, 1].cpu().numpy()
            self.u_initial = torch.tensor(
                func(t=self.t0, x=x, y=y), dtype=torch.float32, device=self.device)
        else:
            self.u_initial = u_initial.to(self.device)
            self.initial_func = None

    def set_boundary_function(self, func):
        self.boundary_func = func

    def _get_bc_values(self, t, pts):
        x = pts[:, 0].cpu().numpy()
        y = pts[:, 1].cpu().numpy()
        return torch.tensor(
            self.boundary_func(t=t, x=x, y=y), dtype=torch.float32, device=self.device)

    def train_td_implicit(self):
        self.PINNS_list  = []
        self.PINNS_times = []
        self.bc_points   = self.get_boundary_points()
        bc0 = self._get_bc_values(self.t0, self.bc_points)
        PINN_old, pde_res = self._train_iter(
            self.xy_initial, self.u_initial,
            initial_condition=True, tol=self.tol_ic,
            xy_bc=self.bc_points, bc=bc0
        )
        self.PINNS_list.append(PINN_old)
        for i in range(1, self.nt):
            t = self.t0 + i * self.dt
            print(f'\n=== Step {i}/{self.nt-1}  t={t:.4f} ===')
            start    = time.perf_counter()
            xy_i     = self.func_distribution(PINN_old, t, pde_res)
            u_0      = PINN_old.forward(xy_i)
            bc_t     = self._get_bc_values(t, self.bc_points)
            PINN_i, pde_res = self._train_iter(
                xy_i, u_0, tol=self.tol, previous_PINN=PINN_old,
                xy_bc=self.bc_points, bc=bc_t
            )
            elapsed  = time.perf_counter() - start
            self.PINNS_list.append(PINN_i)
            self.PINNS_times.append(elapsed)
            print(f'  Done in {elapsed:.2f}s')
            PINN_old = PINN_i

    def _train_iter(self, xy_train, u_train, previous_PINN=None,
                    initial_condition=False, tol=1e-3, xy_bc=None, bc=None):
        xy_train = xy_train.reshape(-1, 2).to(self.device).detach().requires_grad_(True)
        u_train  = u_train.reshape(-1, 1).to(self.device).detach()
        xy_bc    = xy_bc.reshape(-1, 2).to(self.device).detach().requires_grad_(True)
        bc       = bc.reshape(-1, 1).to(self.device).detach()
        PINN = TD_PINNS(device=self.device, x_dim=2, nu=self.nu, dt=self.dt,
                        pde_loss_fn=getattr(self, 'pde_loss_fn', None))
        if previous_PINN:
            PINN.load_state_dict(previous_PINN.state_dict())
        opt   = torch.optim.AdamW(PINN.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode='min', patience=300, factor=0.5, min_lr=self.min_lr)
        print_every = max(1, self.max_epochs // self.n_prints)
        for i in range(self.max_epochs):
            opt.zero_grad()
            if initial_condition:
                loss, l_ic, l_bc, res = PINN.loss_initial(xy_train, u_train, xy_bc, bc)
            else:
                loss, l_pde, l_bc, res = PINN.loss(xy_train, u_train, xy_bc, bc)
            loss.backward(retain_graph=True)
            v = loss.item()
            if v <= tol:
                print(f'  Converged epoch={i}  loss={v:.6f}')
                break
            opt.step()
            sched.step(v)
            if i % print_every == 0:
                print(f'  Epoch {i:6d}: loss={v:.6f}  lr={opt.param_groups[0]["lr"]:.2e}')
        return PINN, res

    def plot_solution(self, PINN, t, nx=50, ny=50):
        xy = self.create_2d_grid(nx, ny)
        with torch.no_grad():
            u = PINN.forward(xy).cpu().numpy().reshape(nx, ny)
        x = xy[:, 0].cpu().numpy().reshape(nx, ny)
        y = xy[:, 1].cpu().numpy().reshape(nx, ny)
        plt.figure(figsize=(7, 5))
        plt.contourf(x, y, u, levels=50, cmap='viridis')
        plt.colorbar(label='u'); plt.xlabel('x'); plt.ylabel('y')
        plt.title(f'2-D Burgers  t={t:.4f}'); plt.show()

    def plot_evolution_subplots(self, nx=40, ny=40, ncols=3):
        n     = len(self.PINNS_list)
        nrows = (n + ncols - 1) // ncols
        xy    = self.create_2d_grid(nx, ny)
        all_u = []
        for P in self.PINNS_list:
            with torch.no_grad():
                all_u.append(P.forward(xy).cpu().numpy())
        vmin = min(u.min() for u in all_u)
        vmax = max(u.max() for u in all_u)
        x = xy[:, 0].cpu().numpy().reshape(nx, ny)
        y = xy[:, 1].cpu().numpy().reshape(nx, ny)
        fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
        axes = np.array(axes).flatten()
        for i, u in enumerate(all_u):
            t_i = self.t0 + i * self.dt
            axes[i].contourf(x, y, u.reshape(nx, ny), levels=30,
                             cmap='viridis', vmin=vmin, vmax=vmax)
            axes[i].set_title(f't={t_i:.3f}')
        for ax in axes[n:]:
            ax.set_visible(False)
        plt.suptitle('2-D Burgers Evolution'); plt.tight_layout(); plt.show()

---
## 1-D Burgers Experiment

Toothsaw (sawtooth) analytic solution on $[0, 2\pi]$ with periodic boundary conditions.

In [ ]:
# ── Analytic solutions ─────────────────────────────────────────────────────

NU_1D = 0.07

def toothsaw(t=0, x=0):
    x  = np.asarray(x, dtype=float)
    nu = NU_1D
    a  = (-4*t + x)**2           / (4*nu*(t + 1))
    b  = (-4*t + x - 2*np.pi)**2 / (4*nu*(t + 1))
    num = ((-8*t + 2*x)            * np.exp(-a)
         + (-8*t + 2*x - 4*np.pi) * np.exp(-b)) / (4*nu*(t + 1))
    den = np.exp(-b) + np.exp(-a)
    return 2*nu * num / den + 4

def sinusoidal_ic(t=0, x=0):
    return -np.sin(np.pi * np.asarray(x, dtype=float))


# ── Setup ──────────────────────────────────────────────────────────────────

X0_1D, XF_1D = 0.0, 2 * np.pi
NX_1D = 100

trainer1d = TrainerBurger1D()
trainer1d.set_device('cpu')
trainer1d.set_simulation_parameters(
    nu=NU_1D, t0=0.0, dt=0.1, nt=7,
    x_bc=torch.Tensor([X0_1D, XF_1D]))
trainer1d.set_training_parameters(
    tol=1e-2, tol_ic=1e-2, max_epochs=80000, n_prints=10, lr=1e-2, min_lr=1e-8)
trainer1d.set_initial_condition(
    x_initial=torch.linspace(X0_1D, XF_1D, NX_1D), func=toothsaw)
trainer1d.set_boundary_function(toothsaw)
trainer1d.set_analytic_solution(toothsaw)


# ── Collocation strategies ─────────────────────────────────────────────────

def lhs_1d(x0, xf, n, device):
    cut = torch.linspace(0, 1, n + 1, device=device)
    u   = torch.rand(n, device=device)
    pts = cut[:-1] + (cut[1:] - cut[:-1]) * u
    return x0 + pts[torch.randperm(n, device=device)] * (xf - x0)

def collocation_uniform(PINN, t, res):
    return torch.linspace(X0_1D, XF_1D, NX_1D).to(trainer1d.device)

def collocation_lhs(PINN, t, res):
    return lhs_1d(X0_1D, XF_1D, NX_1D, trainer1d.device)

def collocation_gradient(PINN, t, res):
    n_a, n_u = NX_1D // 2, NX_1D // 2
    x_test   = torch.linspace(X0_1D, XF_1D, n_a * 3).reshape(-1, 1)
    du_dx, _ = PINN.grad(x_test)
    mag      = du_dx[:, 0].detach().abs()
    idx      = torch.argsort(mag / mag.sum(), descending=True)[:n_a]
    pts = torch.cat([x_test[idx].flatten().to(trainer1d.device),
                     torch.linspace(X0_1D, XF_1D, n_u).to(trainer1d.device)])
    return torch.sort(pts)[0]

def collocation_curvature(PINN, t, res):
    n_a, n_u  = NX_1D // 2, NX_1D // 2
    x_test    = torch.linspace(X0_1D, XF_1D, n_a * 3).reshape(-1, 1)
    _, du_dxx = PINN.grad(x_test)
    mag       = du_dxx[:, 0].detach().abs()
    idx       = torch.argsort(mag / mag.sum(), descending=True)[:n_a]
    pts = torch.cat([x_test[idx].flatten().to(trainer1d.device),
                     torch.linspace(X0_1D, XF_1D, n_u).to(trainer1d.device)])
    return torch.sort(pts)[0]

def collocation_residual(PINN, t, pde_res):
    if pde_res is None:
        return collocation_uniform(PINN, t, pde_res)
    n_a, n_u = NX_1D // 2, NX_1D // 2
    x_test   = torch.linspace(X0_1D, XF_1D, NX_1D).reshape(-1, 1)
    mag      = pde_res[:, 0].detach().abs()
    idx      = torch.argsort(mag / (mag.sum() + 1e-8), descending=True)[:n_a]
    pts = torch.cat([x_test[idx].flatten().to(trainer1d.device),
                     torch.linspace(X0_1D, XF_1D, n_u).to(trainer1d.device)])
    return torch.sort(pts)[0]


# ── Choose collocation & train ─────────────────────────────────────────────
# Options: collocation_uniform, collocation_lhs,
#          collocation_gradient, collocation_curvature, collocation_residual

trainer1d.set_point_collocation_distribution(collocation_curvature)
trainer1d.train_td_implicit()

In [ ]:
trainer1d.plot_evolution()

---
## 2-D Burgers Experiment

Gaussian initial condition on $[-1,1]^2$ with zero Dirichlet boundary conditions.

In [ ]:
# ── Initial / boundary conditions ──────────────────────────────────────────

def gaussian_ic(t=0, x=0, y=0):
    x, y = np.asarray(x), np.asarray(y)
    return np.exp(-(x**2 + y**2) / 0.1)

def zero_bc(t=0, x=0, y=0):
    return np.zeros_like(np.asarray(x))


# ── Setup ──────────────────────────────────────────────────────────────────

NU_2D = 0.01
X0_2D, XF_2D = -1.0, 1.0
Y0_2D, YF_2D = -1.0, 1.0
NX_2D, NY_2D = 20, 20

trainer2d = TrainerBurger2D()
trainer2d.set_device('cpu')
trainer2d.set_simulation_parameters(
    nu=NU_2D, t0=0.0, dt=0.2, nt=6,
    x_range=(X0_2D, XF_2D), y_range=(Y0_2D, YF_2D))
trainer2d.set_training_parameters(
    tol=1e-2, tol_ic=1e-2, max_epochs=30000, n_prints=10, lr=1e-2, min_lr=1e-8)

xy0 = trainer2d.create_2d_grid(NX_2D, NY_2D)
trainer2d.set_initial_condition(xy_initial=xy0, func=gaussian_ic)
trainer2d.set_boundary_function(zero_bc)


# ── Collocation strategies ─────────────────────────────────────────────────

def lhs_2d(x_range, y_range, n, device):
    cut = torch.linspace(0, 1, n + 1, device=device)
    u   = torch.rand(n, 2, device=device)
    a, b = cut[:-1], cut[1:]
    s = a.unsqueeze(1) + (b - a).unsqueeze(1) * u
    for j in range(2):
        s[:, j] = s[torch.randperm(n, device=device), j]
    xmn, xmx = x_range; ymn, ymx = y_range
    s[:, 0] = xmn + s[:, 0] * (xmx - xmn)
    s[:, 1] = ymn + s[:, 1] * (ymx - ymn)
    return s

def collocation_2d_uniform(PINN, t, res):
    return trainer2d.create_2d_grid(NX_2D, NY_2D)

def collocation_2d_lhs(PINN, t, res):
    return lhs_2d((X0_2D, XF_2D), (Y0_2D, YF_2D), NX_2D * NY_2D, trainer2d.device)

def collocation_2d_gradient(PINN, t, res):
    m, n_a = 2, NX_2D * NY_2D // 2
    xy     = trainer2d.create_2d_grid(NX_2D * m, NY_2D * m)
    g, _   = PINN.grad(xy)
    mag    = torch.sqrt(g[:, 0]**2 + g[:, 1]**2).detach()
    idx    = torch.argsort(mag, descending=True)[:n_a]
    n_u    = int(np.sqrt(NX_2D * NY_2D - n_a))
    return torch.cat([xy[idx].detach(),
                      trainer2d.create_2d_grid(n_u, n_u)]).to(trainer2d.device)

def collocation_2d_residual(PINN, t, pde_res):
    if pde_res is None:
        return collocation_2d_uniform(PINN, t, pde_res)
    n_a = NX_2D * NY_2D // 2
    xy  = trainer2d.create_2d_grid(NX_2D, NY_2D)
    mag = pde_res[:, 0].detach().abs()
    idx = torch.argsort(mag / (mag.sum() + 1e-8), descending=True)[:n_a]
    n_u = int(np.sqrt(NX_2D * NY_2D - n_a))
    return torch.cat([xy[idx].detach(),
                      trainer2d.create_2d_grid(n_u, n_u)]).to(trainer2d.device)


# ── Choose collocation & train ─────────────────────────────────────────────
# Options: collocation_2d_uniform, collocation_2d_lhs,
#          collocation_2d_gradient, collocation_2d_residual

trainer2d.set_point_collocation_distribution(collocation_2d_uniform)
trainer2d.train_td_implicit()

In [ ]:
trainer2d.plot_evolution_subplots(nx=40, ny=40, ncols=3)